# ​🅰️​Fundamentos, Ingesta y Limpieza de Datos 

## Problema de negocio
Una empresa de telecomunicaciones pierde clientes que cancelan su suscripción (churn). Quieren predecir el comportamiento de dichos clientes para poder retenerlos, por lo que se debe determinar qué características se asocian con la cancelación del servicio. A partir de ahí, se puede anticipar qué clientes están en riesgo e implementar medidas preventivas.  


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Planteo de hipótesis e identificación de variable objetivo

In [19]:
df.shape

(7043, 21)

In [20]:
df.head(25)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,...,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.10,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,...,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,...,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [22]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [23]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Hipótesis analíticas

### H1
Los clientes con contrato mes a mes cancelan más que los de contratos de 1 o 2 años. El contrato mes a mes no tiene penalidad por cancelar, así que irse es más fácil. Los anuales y bienales son decisiones a largo plazo, que conllevan un nivel de ponderación distinta a la hora de rescindir el contrato.

### H2
A menor antigüedad (`tenure`), mayor probabilidad de cancelar la suscripción. Los primeros meses son clave, ya que un cliente nuevo no está acostumbrado al servicio, ni siente necesariamente que le deba fidelidad alguna. 

### H3
Los clientes con cargos mensuales altos cancelan más. A mayor precio, más pesa la decisión de seguir pagando.

### H4
Quienes pagan con cheque electrónico cancelan más que quienes tienen débito automático. Es distinto cómo se procesa el gasto consciente, que el gasto automatizado. Con el pago manual, el cliente tiene que autorizar el pago cada vez, así que la decisión de seguir con el servicio se repite constantemente. Con el débito automático, el pago ocurre sin que intervenga y la cancelación requiere una acción deliberada.

## Variable objetivo: `Churn`.
Es categórica nominal binaria (Yes/No) e indica si el cliente canceló el servicio.

## Clasificación del Dataset y Variables
El dataset es de tipo estructurado, se presenta en formato CSV.
### Clasificación teórica de las variables
- **Cualitativas nominales:** `customerID` (identificador único), `gender`, `SeniorCitizen`, `Partner`, `Dependents`, `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling`, `PaymentMethod` y `Churn`

- **Cualitativas ordinales:** `Contract`

- **Cuantitativas discretas:** `tenure`

- **Cuantitativas continuas:** `MonthlyCharges`, `TotalCharges`

In [16]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

`SeniorCitizen`: pandas la lee como **int64**, pero es cualitativa nominal (0/1 = no/sí).
`TotalCharges`: pandas la lee como **texto**, pero es cuantitativa continua, porque tiene cadenas " ".
`Contract`: pandas la lee como **texto** y no refleja el orden de una variable ordinal.
`customerID`: **texto**, coincide con su clasificación, pero es un identificador.
El resto coincide: las Yes/No como **texto**, `tenure` como entero (discreta) y `MonthlyCharges` como **float64** (continua).

## Limpieza y Calidad de Datos
Se trabaja sobre una copia (`df_clean`) para conservar el dataset original.

In [17]:
df_clean = df.copy()

### Tratamiento de valores faltantes y cadenas anómalas
`df.isnull().sum()` no encontró valores nulos, pero `df.info()` mostró que `TotalCharges` es de tipo texto, aunque debería ser numérica. Se buscan cadenas vacías o compuestas solo por espacios en todas las columnas de texto.

In [18]:
columnas_texto = df_clean.select_dtypes(exclude='number').columns
cadenas_vacias = pd.Series({col: (df_clean[col].str.strip() == '').sum() for col in columnas_texto})
cadenas_vacias[cadenas_vacias > 0]

TotalCharges    11
dtype: int64

Solo `TotalCharges` tiene cadenas anómalas (`" "`). Se registran 11 entradas vacías. Se revisan las filas correspondientes.

In [24]:
filas_vacias = df_clean['TotalCharges'].str.strip() == ''
df_clean.loc[filas_vacias, ['tenure', 'Contract', 'MonthlyCharges', 'TotalCharges']]

,tenure,Contract,MonthlyCharges,TotalCharges
488,0,Two year,52.55,
753,0,Two year,20.25,
936,0,Two year,80.85,
1082,0,Two year,25.75,
1340,0,Two year,56.05,
3331,0,Two year,19.85,
3826,0,Two year,25.35,
4380,0,Two year,20.00,
5218,0,One year,19.70,
6670,0,Two year,73.35,


In [25]:
print('Clientes con tenure = 0:          ', (df_clean['tenure'] == 0).sum())
print('Clientes con TotalCharges vacío:  ', filas_vacias.sum())
print('Coinciden ambas condiciones:      ', (filas_vacias & (df_clean['tenure'] == 0)).sum())

Clientes con tenure = 0:           11
Clientes con TotalCharges vacío:   11
Coinciden ambas condiciones:       11


Los 11 registros con `TotalCharges` vacío son los 11 clientes con `tenure = 0`: clientes recién dados de alta a los que todavía no se les facturó. No es un error de carga, el valor falta porque aún no hay nada facturado.

Primero se convierten las cadenas vacías a `NaN` al pasar la columna a numérica, y luego se reemplaza por `0`, que es el gasto acumulado real de un cliente sin meses de servicio.

In [27]:
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce')
print('Valores nulos en TotalCharges después de convertir:', df_clean['TotalCharges'].isnull().sum())

Valores nulos en TotalCharges después de convertir: 11


In [29]:
df_clean['TotalCharges'] = df_clean['TotalCharges'].fillna(0)
print('Valores nulos en TotalCharges después del reemplazo:', df_clean['TotalCharges'].isnull().sum())
print('Valores nulos en todo el dataset:                ', df_clean.isnull().sum().sum())

Valores nulos en TotalCharges después del reemplazo: 0
Valores nulos en todo el dataset:                 0


### Adecuación de tipos de datos
Se corrigen las diferencias detectadas al contrastar la clasificación teórica con los `dtypes` de pandas:
- **`TotalCharges`**: pasó de texto a `float64` en el paso anterior (cuantitativa continua).
- **`Contract`**: pasa de texto a categórica ordenada (`Month-to-month` < `One year` < `Two year`), para que pandas refleje que es una variable ordinal.
- **`SeniorCitizen`**: se mantiene como `int64` (0/1). Teóricamente es cualitativa nominal, pero la codificación 0/1 ya es la que necesitan los modelos; pasarla a texto obligaría a volver a codificarla en el preprocesamiento.
- **`Churn`** y las demás variables Yes/No se mantienen como texto; se codifican en la etapa de preprocesamiento para modelado.

In [30]:
orden_contrato = ['Month-to-month', 'One year', 'Two year']
df_clean['Contract'] = pd.Categorical(df_clean['Contract'], categories=orden_contrato, ordered=True)

df_clean[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Contract']].dtypes

SeniorCitizen        int64
tenure               int64
MonthlyCharges     float64
TotalCharges       float64
Contract          category
dtype: object

In [31]:
df_clean['Contract'].cat.categories

Index(['Month-to-month', 'One year', 'Two year'], dtype='str')

## Manipulación de datos
### Selección: filtrado con múltiples condiciones
**Segmento de riesgo potencial:** clientes con contrato mes a mes que, además, pagan con *Electronic check* o tienen 12 meses o menos de antigüedad.

In [34]:
segmento_riesgo = df_clean[
    (df_clean['Contract'] == 'Month-to-month') &
    ((df_clean['PaymentMethod'] == 'Electronic check') | (df_clean['tenure'] <= 12))
]

total_mensual = df_clean['MonthlyCharges'].sum()
print(f"Clientes en el segmento:            {len(segmento_riesgo)} ({len(segmento_riesgo) / len(df_clean):.1%} del total)")
print(f"Facturación mensual del segmento:   {segmento_riesgo['MonthlyCharges'].sum():,.2f} "
      f"({segmento_riesgo['MonthlyCharges'].sum() / total_mensual:.1%} del total)")

Clientes en el segmento:            2890 (41.0% del total)
Facturación mensual del segmento:   188,745.50 (41.4% del total)


Si las hipótesis se confirman, una parte importante del ingreso mensual depende de clientes con un perfil de riesgo.

### Transformación básica
**Renombrado de columnas:** todas las columnas usan CamelCase salvo `customerID`, `gender` y `tenure`. Se unifican para mantener un criterio de nombres consistente.

In [35]:
nombres_nuevos = {
    'customerID': 'CustomerID',
    'gender': 'Gender',
    'tenure': 'Tenure',
}
df_clean = df_clean.rename(columns=nombres_nuevos)
df_clean.columns.tolist()

['CustomerID',
 'Gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'Tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

**Eliminación de atributos irrelevantes:** `CustomerID` es un identificador único por cliente; no describe ninguna característica y no aporta información para explicar ni predecir la cancelación. Se elimina.

In [36]:
df_clean = df_clean.drop(columns=['CustomerID'])
print('Columnas restantes:', df_clean.shape[1])

Columnas restantes: 20


**Columnas calculadas:**
- `AvgMonthlyCharge` = `TotalCharges / Tenure`: cargo mensual promedio histórico.
- `ChargeDiff` = `MonthlyCharges − AvgMonthlyCharge`: diferencia entre lo que paga hoy y lo que pagó en promedio.
- `NumServices`: cantidad de servicios contratados (telefonía, internet y servicios adicionales).

In [38]:
# Cargo mensual promedio histórico: lo que el cliente pagó en promedio por mes.
# Con Tenure = 0 no hay historial: se usa el cargo mensual actual para no dividir por cero.
df_clean['AvgMonthlyCharge'] = (
    df_clean['TotalCharges'] / df_clean['Tenure'].replace(0, np.nan)
).fillna(df_clean['MonthlyCharges']).round(2)

# Diferencia entre el cargo actual y el promedio histórico.
df_clean['ChargeDiff'] = (df_clean['MonthlyCharges'] - df_clean['AvgMonthlyCharge']).round(2)

# Cantidad de servicios contratados: teléfono, internet y servicios adicionales.
servicios_adicionales = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                         'TechSupport', 'StreamingTV', 'StreamingMovies']
df_clean['NumServices'] = (
    (df_clean['PhoneService'] == 'Yes').astype(int)
    + (df_clean['InternetService'] != 'No').astype(int)
    + (df_clean[servicios_adicionales] == 'Yes').sum(axis=1)
)

df_clean[['Tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'ChargeDiff', 'NumServices']].head()

,Tenure,MonthlyCharges,TotalCharges,AvgMonthlyCharge,ChargeDiff,NumServices
0,1,29.85,29.85,29.85,0.00,2
1,34,56.95,1889.50,55.57,1.38,4
2,2,53.85,108.15,54.08,-0.23,4
3,45,42.30,1840.75,40.91,1.39,4
4,2,70.70,151.65,75.82,-5.12,2


In [39]:
df_clean[['AvgMonthlyCharge', 'ChargeDiff', 'NumServices']].describe().round(2)

,AvgMonthlyCharge,ChargeDiff,NumServices
count,7043.00,7043.00,7043.00
mean,64.76,-0.00,4.15
std,30.19,2.61,2.31
min,13.78,-18.90,1.00
25%,35.94,-1.16,2.00
50%,70.34,0.00,4.00
75%,90.17,1.14,6.00
max,121.40,19.12,9.00


In [40]:
print(f"Clientes que hoy pagan más que su promedio histórico (ChargeDiff > 5): {(df_clean['ChargeDiff'] > 5).mean():.1%}")
print(f"Clientes que hoy pagan menos que su promedio histórico (ChargeDiff < -5): {(df_clean['ChargeDiff'] < -5).mean():.1%}")
print(f"Cargo mensual promedio según cantidad de servicios:")
df_clean.groupby('NumServices')['MonthlyCharges'].mean().round(2)

Clientes que hoy pagan más que su promedio histórico (ChargeDiff > 5): 3.1%
Clientes que hoy pagan menos que su promedio histórico (ChargeDiff < -5): 3.1%
Cargo mensual promedio según cantidad de servicios:


NumServices
1     20.27
2     42.00
3     61.70
4     69.36
5     78.73
6     86.31
7     93.64
8     99.43
9    104.63
Name: MonthlyCharges, dtype: float64

- `ChargeDiff` demuestra que los precios fueron estables, lo que también confirma que `TotalCharges` ≈ `Tenure × MonthlyCharges`.
- `NumServices` permite distinguir si un cargo alto se debe a muchos servicios o a un precio alto por servicio.

In [41]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Gender            7043 non-null   str     
 1   SeniorCitizen     7043 non-null   int64   
 2   Partner           7043 non-null   str     
 3   Dependents        7043 non-null   str     
 4   Tenure            7043 non-null   int64   
 5   PhoneService      7043 non-null   str     
 6   MultipleLines     7043 non-null   str     
 7   InternetService   7043 non-null   str     
 8   OnlineSecurity    7043 non-null   str     
 9   OnlineBackup      7043 non-null   str     
 10  DeviceProtection  7043 non-null   str     
 11  TechSupport       7043 non-null   str     
 12  StreamingTV       7043 non-null   str     
 13  StreamingMovies   7043 non-null   str     
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   str     
 16  PaymentMethod     7043 non-null   s

`df_clean` es el DataFrame resultante de la limpieza de datos realizada, sin cadenas anómalas ni nulos, con tipos corregidos, columnas renombradas, sin el identificador y con tres columnas calculadas. Es la entrada para el análisis exploratorio y el modelado.